# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page is likely to have a good click-through rate if it gets a decent number of impressions and has a proper amount of content. If it gets enough impressions but people still aren't clicking, that's worth flagging as a signal problem.

Reason Codes:

good_ctr — the page is getting impressions and its CTR is at or above average.

low_ctr_but_visible — the page is getting enough impressions, but CTR is below average (can be flagged).

not_enough_visibility — the page isn't getting enough impressions to judge CTR meaningfully yet.

missing_content_data — word count (or other content info) is missing for this page, so the rule can't fully judge it.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#load the data
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/peddikotlahimani/Flyrank-internship/main/data/raw/content_refresh_anonymized.csv")

#what "average CTR" means for this data,we have something to compare each page against
average_ctr = df["ctr"].mean()
print("Average CTR:", average_ctr)

#rule, as a function
def figure_out_reason(row):
    if pd.isna(row["word_count"]):
        return "missing_content_data"
    elif row["impressions_90d"] < 100:
        return "not_enough_visibility"
    elif row["ctr"] >= average_ctr:
        return "good_ctr"
    else:
        return "low_ctr_but_visible"

#apply the rule to every row
df["reason_code"] = df.apply(figure_out_reason, axis=1)

#turn each reason code into simple score number
score_map = {
    "good_ctr": 3,
    "low_ctr_but_visible": 2,
    "not_enough_visibility": 1,
    "missing_content_data": 0
}
df["baseline_score"] = df["reason_code"].map(score_map)

#sort so best pages are at the top, and give each a rank number
ranked_queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

#save it as a CSV file
import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

#confirm it saved correctly
print("Saved successfully!")
print("File exists:", os.path.exists("work/outputs/baseline_action_score.csv"))
ranked_queue.head(10)


Average CTR: 0.5107333333333334
Saved successfully!
File exists: True


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,reason_code,baseline_score,rank
0,content_42fb2cad9ecf,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3969.0,24527.0,...,3.43,2.61,0.00,good,page_1,up,277.8,good_ctr,3,1
1,content_4bb1a7b49efb,client_0b918943df,10.0,0.14,LOW,0.00,keyword article,transactional,1552.0,10368.0,...,0.00,22.22,0.00,low,page_1,down,-27.8,good_ctr,3,2
2,content_4998a1c76243,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,3146.0,20309.0,...,0.00,0.00,0.00,moderate,page_1,up,90.7,good_ctr,3,3
3,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,5.88,4.55,0.00,good,striking,down,-41.4,good_ctr,3,4
4,content_887020f20b5e,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,4124.0,26315.0,...,6.58,3.31,2.63,good,page_1,stable,12.3,good_ctr,3,5
5,content_7600c510155a,client_6208ef0f77,10.0,0.06,LOW,9.56,keyword article,commercial,5583.0,37432.0,...,1.08,1.33,0.76,excellent,page_3_5,down,-30.1,good_ctr,3,6
6,content_4cb8ceca114e,client_4e07408562,30.0,0.37,MEDIUM,0.00,keyword article,informational,2837.0,17468.0,...,9.09,10.94,0.00,good,page_1,stable,2.7,good_ctr,3,7
7,content_ca219993f697,client_624b60c58c,NaN,NaN,NaN,NaN,feedly article,NaN,4830.0,33129.0,...,0.00,50.00,0.00,moderate,page_1,down,-64.3,good_ctr,3,8
8,content_a519180b7a3f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7065.0,46083.0,...,1.72,4.68,0.00,good,striking,stable,-8.3,good_ctr,3,9
9,content_95d488a56079,client_f74efabef1,0.0,0.00,LOW,0.00,keyword article,informational,2937.0,21281.0,...,0.00,4.17,0.00,moderate,striking,stable,4.5,good_ctr,3,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
top_20 = ranked_queue.head(20)
top_20[["content_id", "reason_code", "baseline_score", "impressions_90d", "ctr", "word_count"]]

,content_id,reason_code,baseline_score,impressions_90d,ctr,word_count
0,content_42fb2cad9ecf,good_ctr,3,7228,1.76,3969.0
1,content_4bb1a7b49efb,good_ctr,3,242,3.72,1552.0
2,content_4998a1c76243,good_ctr,3,499,0.60,3146.0
3,content_304f48230142,good_ctr,3,3803,0.76,3221.0
4,content_887020f20b5e,good_ctr,3,4234,1.89,4124.0
5,content_7600c510155a,good_ctr,3,32767,0.58,5583.0
6,content_4cb8ceca114e,good_ctr,3,6017,0.55,2837.0
7,content_ca219993f697,good_ctr,3,322,1.86,4830.0
8,content_a519180b7a3f,good_ctr,3,10934,0.55,7065.0
9,content_95d488a56079,good_ctr,3,564,0.89,2937.0


| # | Page | CTR | Impressions | Word Count | Action | Confidence | What would make it wrong |
|---|------|-----|-------------|------------|--------|------------|---------------------------|
| 1 | 42fb2cad | 1.76 | 7,228 | 3,969 | Protect | High | Large sample, comfortably above average — solid |
| 2 | 4bb1a7b4 | 3.72 | 242 | 1,552 | Protect | Medium | Highest CTR here, but only 242 impressions — a few extra clicks could swing this a lot |
| 3 | 4998a1c7 | 0.60 | 499 | 3,146 | Protect | Low | Barely above average CTR (0.51) — thin margin, easily flips with small changes |
| 4 | 304f4823 | 0.76 | 3,803 | 3,221 | Protect | High | Large sample, safely above average |
| 5 | 887020f2 | 1.89 | 4,234 | 4,124 | Protect | High | Large sample, well above average |
| 6 | 7600c510 | 0.58 | 32,767 | 5,583 | Protect | Low | Huge sample, but CTR is only just above average — real edge is tiny |
| 7 | 4cb8ceca | 0.55 | 6,017 | 2,837 | Protect | Low | Large sample but CTR barely clears average — marginal win, not a strong signal |
| 8 | ca219993 | 1.86 | 322 | 4,830 | Protect | Medium | Decent CTR but sample is small; also missing keyword/intent data |
| 9 | a519180c | 0.55 | 10,934 | 7,065 | Protect | Low | Large sample, but CTR barely above average — weak margin |
| 10 | 95d488a5 | 0.89 | 564 | 2,937 | Protect | Medium | Moderate sample; CTR could shift with more data |
| 11 | 3223b164 | 1.27 | 157 | 3,257 | Protect | Low | Very small sample (157) — CTR here is not very trustworthy yet |
| 12 | 995627a1 | 0.86 | 349 | 2,801 | Protect | Medium | Small-ish sample; watch before fully trusting |
| 13 | 0d748c48 | 0.55 | 2,538 | 2,692 | Protect | Low | CTR barely clears average — marginal, not a strong win |
| 14 | ff10702c | 0.59 | 675 | 2,702 | Protect | Low | Small sample and only just above average — weak evidence |
| 15 | 6109a89b | 0.54 | 2,601 | 2,899 | Protect | Low | CTR nearly identical to average — essentially tied, not truly "good" |
| 16 | f723386d | 1.12 | 178 | 2,855 | Protect | Low | Very small sample (178) — easily noisy |
| 17 | a6b49aac | 0.64 | 6,739 | 6,699 | Protect | Medium | Large sample, modest margin above average |
| 18 | 32be9f63 | 0.52 | 964 | 2,555 | Protect | Low | CTR is essentially equal to the average (0.51) — barely qualifies |
| 19 | dfa5f166 | 0.60 | 7,036 | 2,267 | Protect | Low | Large sample but small margin above average |
| 20 | b560ec37 | 0.90 | 111 | 675 | Protect | Low | Smallest sample here (111) AND thinnest content (675 words) — weakest pick on the list |

## 4. Weak picks + leakage check
*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: 6 of my top 20 pages have a CTR that's only barely above the average (0.51). My rule just checks "above or below average," so it can't tell the difference between a page that's slightly better and a page that's a lot better,they all get treated the same. Row 20 is the weakest pick: it has the fewest impressions (111) and the least content (675 words) of anyone in the top 20, but it still made the list just because its CTR was technically above average.

Leakage check: My rule only used word_count, impressions_90d, and ctr to build the score. I did not use trend_direction or trend_pct anywhere as they give direct answers. I also skipped provider_used and model_used since the data dictionary says those aren't allowed as features either. Everything I used is information we already know.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
used_columns = ["word_count", "impressions_90d", "ctr"]
not_allowed = ["trend_direction", "trend_pct", "provider_used", "model_used"]

leaked = any(col in used_columns for col in not_allowed)
print("Did I use a column I wasn't supposed to?", leaked)

Did I use a column I wasn't supposed to? False


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes ] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.